# 合成ナノシート インスタンスセグメンテーション デモ
# Synthetic Nanosheet Instance Segmentation Demo

---

## このノートブックで学ぶこと

合成ナノシート画像を使って、画像領域分割の基本的な流れを体験します。

扱うタスクは、画像中の個々のナノシートを分けて検出する **instance segmentation** です。

このデモでは、**3つの難易度レベル**（Easy / Mid / Hard）で、以下の3つの手法を比較します：

### 1. SAM (ViT-H) — ゼロショットセグメンテーション
Segment Anything Model (ViT-H) の Automatic Mask Generator (AMG) を使用。
タスク固有の学習なしで、あらゆる画像に適用できる基盤モデルです。

### 2. YOLO-seg (pretrained) — 事前学習のみ
COCO データセットで学習済みの YOLOv11s-seg をそのまま使用。
一般的な物体検出モデルではナノシートを認識できないことを示します。

### 3. YOLO-seg (trained) — 合成データで学習済み
各難易度の合成ナノシート画像100枚で fine-tuning した YOLOv11s-seg。
タスク特化の学習データの価値を示します。

---

**重要：** このpublicデモには、実際の実験TEM画像、手動アノテーション、未公開の実験データは含まれていません。
すべての画像、マスク、モデルweights、予測結果は、教材用に生成した人工データに基づいています。

---

### 学習の流れ

1. 合成データ生成（生成の仕組みを理解する）
2. 正解マスクの確認
3. 3手法 × 3難易度の予測結果を読み込み
4. 評価指標を算出する
5. 難易度ごとの比較可視化
6. どの方法がいいかを比較する

## Step 1: GitHubから教材を取得する

In [ ]:
import os

repo_dir = "/content/nanosheet-segmentation-colab-demo"

if not os.path.exists(repo_dir):
    !git clone https://github.com/fanfanfuzzy/nanosheet-segmentation-colab-demo.git
else:
    print("Repository already exists. Skipping clone.")

%cd /content/nanosheet-segmentation-colab-demo

## Step 2: 必要なライブラリをインストールする

In [ ]:
!pip install -r requirements.txt -q

## Step 3: 合成ナノシート画像を生成する

ここでは、物理モデル（Beer-Lambert則に基づく透過モデル）を使って、
ナノシートの合成画像を生成します。

難易度設定（`configs/`）によってノイズ・コントラスト・境界の明瞭さが変わります：
- `synthetic_easy.yaml`: 高コントラスト・低ノイズ（SNR ≈ 2.7）
- `synthetic_mid.yaml`: 中程度（SNR ≈ 1.7）
- `synthetic_hard.yaml`: 低コントラスト・高ノイズ（SNR ≈ 1.1）

In [ ]:
!python src/generate_synthetic_nanosheets.py \
    --config configs/synthetic_mid.yaml \
    --num-images 10 \
    --output-dir outputs/synthetic_demo

## Step 4: 正解マスクを可視化する

生成した画像と各ナノシートの正解マスク（instance mask）を重ねて表示します。
色が異なる領域が、それぞれ個別のナノシートです。

In [ ]:
!python src/visualize_dataset.py \
    --input-dir outputs/synthetic_demo \
    --output-dir outputs/figures

In [ ]:
from IPython.display import Image, display
display(Image("outputs/figures/sample_dataset.png"))

## Step 5: 3手法 × 3難易度の評価

ここでは、事前に計算された予測結果を使って、各手法を各難易度で評価します。

### 予測手法
1. **SAM (ViT-H)** — Automatic Mask Generator (AMG) でゼロショット推論
2. **YOLO-seg pretrained** — COCO 学習済みモデル（fine-tuning なし）
3. **YOLO-seg trained** — 各難易度の合成データ100枚で学習済み

### 評価指標の意味

| 指標 | 意味 |
|------|------|
| `instance_recall` | 正解のナノシートのうちモデルが見つけられた割合 |
| `instance_precision` | モデルが出した予測のうち、正解だった割合 |
| `instance_f1` | RecallとPrecisionの調和平均 |
| `mean_best_iou` | 各正解に対して最もよくなった予測とのIoUの平均 |
| `semantic_iou` | 個別を区別せず前景全体としてのIoU |

In [ ]:
import subprocess

difficulties = ["easy", "mid", "hard"]
methods = [
    ("sam", "predictions_sam"),
    ("yolo_pretrained", "predictions_yolo_pretrained"),
    ("yolo_trained", "predictions_yolo_trained"),
]

for diff in difficulties:
    for method_name, pred_subdir in methods:
        gt_dir = f"demo_assets/{diff}/ground_truth"
        pred_dir = f"demo_assets/{diff}/{pred_subdir}"
        output_csv = f"outputs/metrics_{method_name}_{diff}.csv"
        cmd = [
            "python", "src/evaluate_predictions.py",
            "--gt-dir", gt_dir,
            "--pred-dir", pred_dir,
            "--method-name", f"{method_name}_{diff}",
            "--output", output_csv,
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print(f"Error evaluating {method_name} on {diff}:")
            print(result.stderr)
        else:
            # Print average line
            for line in result.stdout.split("\n"):
                if "instance_recall" in line or "instance_precision" in line or "mean_best_iou" in line:
                    print(f"  [{diff}/{method_name}] {line.strip()}")

print("\nAll evaluations complete.")

## Step 6: 難易度ごとの評価指標を比較する

3つの難易度 × 3つの手法の評価結果を表にまとめます。

In [ ]:
import pandas as pd
from IPython.display import display

difficulties = ["easy", "mid", "hard"]
methods = ["sam", "yolo_pretrained", "yolo_trained"]

rows = []
for diff in difficulties:
    for method in methods:
        csv_path = f"outputs/metrics_{method}_{diff}.csv"
        try:
            df = pd.read_csv(csv_path)
            avg_row = df[df["image"] == "AVERAGE"].iloc[0]
            rows.append({
                "difficulty": diff,
                "method": method,
                "instance_recall": f"{avg_row['instance_recall']:.3f}",
                "instance_precision": f"{avg_row['instance_precision']:.3f}",
                "instance_f1": f"{avg_row['instance_f1']:.3f}",
                "mean_best_iou": f"{avg_row['mean_best_iou']:.3f}",
                "semantic_iou": f"{avg_row['semantic_iou']:.3f}",
            })
        except Exception as e:
            print(f"Warning: could not load {csv_path}: {e}")

summary_df = pd.DataFrame(rows)
display(summary_df)
summary_df.to_csv("outputs/multi_difficulty_comparison.csv", index=False)
print("\nSaved to outputs/multi_difficulty_comparison.csv")

## Step 7: 難易度ごとの性能比較（棒グラフ）

難易度が上がるにつれて：
- **SAM** のrecallが大幅に低下する（ノイズに弱い）
- **YOLO-seg trained** は難易度に応じた学習データで訓練されているため、比較的安定

この差が、タスク特化の学習データの価値を示しています。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

summary_df = pd.read_csv("outputs/multi_difficulty_comparison.csv")

difficulties = ["easy", "mid", "hard"]
methods = ["sam", "yolo_pretrained", "yolo_trained"]
method_labels = ["SAM (ViT-H)", "YOLO pretrained", "YOLO trained"]
colors = ["#2196F3", "#FF9800", "#4CAF50"]
metrics_to_plot = ["instance_recall", "instance_f1", "mean_best_iou", "semantic_iou"]
metric_labels = ["Instance Recall", "Instance F1", "Mean Best IoU", "Semantic IoU"]

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(5 * len(metrics_to_plot), 5))
x = np.arange(len(difficulties))
width = 0.25

for ax, metric, label in zip(axes, metrics_to_plot, metric_labels):
    for i, (method, method_label, color) in enumerate(zip(methods, method_labels, colors)):
        vals = []
        for diff in difficulties:
            row = summary_df[(summary_df["difficulty"] == diff) & (summary_df["method"] == method)]
            vals.append(float(row[metric].iloc[0]) if len(row) > 0 else 0)
        ax.bar(x + i * width, vals, width, label=method_label, color=color, alpha=0.85)
    ax.set_xlabel("Difficulty")
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.set_xticks(x + width)
    ax.set_xticklabels([d.capitalize() for d in difficulties])
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)

plt.suptitle("Segmentation Performance by Difficulty Level", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/multi_difficulty_barplot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/multi_difficulty_barplot.png")

## Step 8: テスト画像ごとの可視化比較

各難易度から代表的なテスト画像を選び、5列の比較グリッドを生成します：
`Original | Ground Truth | SAM | YOLO pretrained | YOLO trained`

インスタンス数の違いに注目してください。

In [ ]:
for diff in ["easy", "mid", "hard"]:
    print(f"\n=== {diff.upper()} ===")
    !python src/visualize_comparison.py \
        --image-dir demo_assets/{diff}/test_images \
        --gt-dir demo_assets/{diff}/ground_truth \
        --zero-shot-dir demo_assets/{diff}/predictions_sam \
        --pretrained-dir demo_assets/{diff}/predictions_yolo_pretrained \
        --trained-dir demo_assets/{diff}/predictions_yolo_trained \
        --output-dir outputs/comparison_visual_{diff} \
        --max-images 3

In [ ]:
from IPython.display import Image, display

for diff in ["easy", "mid", "hard"]:
    print(f"\n{'='*60}")
    print(f"  {diff.upper()} difficulty")
    print(f"{'='*60}")
    summary_path = f"outputs/comparison_visual_{diff}/comparison_summary.png"
    if os.path.exists(summary_path):
        display(Image(summary_path))

## Step 9: 発展 — YOLOの短時間学習を試す（任意）

---

**このセルは実行しなくてもワークショップは完結します。**

GPUランタイムが利用可能な場合のみ、短時間のYOLO学習を体験できます。
実際には数分かかる場合があります。

```
ランタイム → ランタイムのタイプを変更 → GPU を選択
```

In [ ]:
# Optional: run a very short YOLO-seg training demo.
# This cell requires a GPU runtime and may take a few minutes.
# Uncomment the lines below to try.

# !pip install ultralytics -q
# !yolo segment train \
#     model=yolo11n-seg.pt \
#     data=demo_assets/yolo_dataset.yaml \
#     epochs=3 \
#     imgsz=512 \
#     batch=4

## Step 10: まとめ

このデモで、以下を学びました：

1. **合成ナノシート画像の生成** — Beer-Lambert物理モデルに基づく合成データ作成
2. **3つの難易度レベル** — Easy（SNR≈2.7）/ Mid（SNR≈1.7）/ Hard（SNR≈1.1）での比較
3. **SAM (ViT-H) ゼロショット** — 学習なしでAutomatic Mask Generatorを使う手法
4. **YOLO-seg (pretrained)** — 一般的な事前学習モデルはナノシートを認識できない
5. **YOLO-seg (trained)** — 合成データ100枚で学習したモデルの性能
6. **難易度による性能変化** — ノイズが増えるとSAMの性能が大幅に低下し、学習済みモデルの価値が際立つ

---

### 実際の研究では

実際の研究では、合成データで高性能だったモデルも実際の画像に適用すると性能が
低下する場合があります。
この差は **sim2real gap** と呼ばれます。

本publicデモでは、実験データそのものは公開せず、合成データを使って同じ評価の流れ
だけを体験しました。

---

### AI支援コーディングについて

このリポジトリの一部はAIコーディングアシスタント（Devin）を使って作成されています。

重要なのはAIにコードを書かせることよりも、
**AIが行った変更をGitHub上で確認し、安全に取り込むこと**です。

Pull Requestの「Files changed」タブで差分を確認する習慣をつけましょう。